# Evaluation Metrics

Implement accuracy, precision, recall, F1, and confusion matrix from scratch for binary classification. Show the precision/recall tradeoff via a threshold sweep and plot a ROC curve with AUC.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


In [2]:
# sklearn is an optional dependency — validate against it when available,
# otherwise fall back to hand-computed reference values.
try:
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix as sk_confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )
    SKLEARN_AVAILABLE = True
    print("sklearn available — will validate against sklearn.metrics")
except ImportError:
    SKLEARN_AVAILABLE = False
    print("sklearn not installed — will validate against hand-computed reference values")


sklearn not installed — will validate against hand-computed reference values


## Hand-computed reference values

We define a tiny 8-sample fixture whose ground-truth metrics can be verified mentally. These are used as the validation target when sklearn is not installed.

```
y_true = [1, 1, 1, 1, 0, 0, 0, 0]  (4 positives, 4 negatives)
y_pred = [1, 1, 0, 0, 0, 0, 1, 1]  (TP=2, FN=2, TN=2, FP=2)
```

From the confusion matrix:
- TP = 2, FP = 2, FN = 2, TN = 2
- Accuracy = (TP + TN) / N = 4/8 = 0.5
- Precision = TP / (TP + FP) = 2/4 = 0.5
- Recall = TP / (TP + FN) = 2/4 = 0.5
- F1 = 2 * P * R / (P + R) = 0.5

In [3]:
# Tiny fixture for deterministic hand-computed validation
y_true_tiny = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0], dtype=torch.long)
y_pred_tiny = torch.tensor([1, 1, 0, 0, 0, 0, 1, 1], dtype=torch.long)
# Expected: TP=2, FP=2, FN=2, TN=2 -> acc=0.5, prec=0.5, rec=0.5, f1=0.5
REF = {"accuracy": 0.5, "precision": 0.5, "recall": 0.5, "f1": 0.5}
print("Reference values:", REF)


Reference values: {'accuracy': 0.5, 'precision': 0.5, 'recall': 0.5, 'f1': 0.5}


## From-scratch binary classification metrics

We work from the confusion matrix: a 2×2 table of true/false positives/negatives.

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

- **Accuracy**: overall fraction correct. Misleading under class imbalance.
- **Precision**: among predicted positives, fraction that are truly positive. Answers: 'when we raise an alarm, how often is it real?'
- **Recall** (sensitivity, true positive rate): among actual positives, fraction detected. Answers: 'how many real cases did we catch?'
- **F1**: harmonic mean of precision and recall. Penalizes extreme tradeoffs.

In [4]:
def confusion_matrix_binary(
    y_true: torch.Tensor, y_pred: torch.Tensor
) -> tuple[int, int, int, int]:
    """Return (TP, FP, FN, TN) for binary {0, 1} tensors."""
    tp = int(((y_pred == 1) & (y_true == 1)).sum().item())
    fp = int(((y_pred == 1) & (y_true == 0)).sum().item())
    fn = int(((y_pred == 0) & (y_true == 1)).sum().item())
    tn = int(((y_pred == 0) & (y_true == 0)).sum().item())
    return tp, fp, fn, tn


def accuracy(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """Fraction of correctly classified samples."""
    return (y_pred == y_true).float().mean().item()


def precision_score_scratch(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """TP / (TP + FP). Returns 0.0 if no positive predictions."""
    tp, fp, fn, tn = confusion_matrix_binary(y_true, y_pred)
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def recall_score_scratch(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """TP / (TP + FN). Returns 0.0 if no actual positives."""
    tp, fp, fn, tn = confusion_matrix_binary(y_true, y_pred)
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def f1_score_scratch(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    """Harmonic mean of precision and recall: 2*P*R / (P + R)."""
    p = precision_score_scratch(y_true, y_pred)
    r = recall_score_scratch(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


# Validate on tiny fixture (hand-computed reference)
metrics_tiny = {
    "accuracy":  accuracy(y_true_tiny, y_pred_tiny),
    "precision": precision_score_scratch(y_true_tiny, y_pred_tiny),
    "recall":    recall_score_scratch(y_true_tiny, y_pred_tiny),
    "f1":        f1_score_scratch(y_true_tiny, y_pred_tiny),
}
print("Computed metrics on tiny fixture:")
for k, v in metrics_tiny.items():
    print(f"  {k:12s}: {v:.4f}  (expected {REF[k]:.4f})")
    assert abs(v - REF[k]) < 1e-6, f"{k} mismatch: {v} vs {REF[k]}"
print("All metrics match hand-computed reference values ✓")


Computed metrics on tiny fixture:
  accuracy    : 0.5000  (expected 0.5000)
  precision   : 0.5000  (expected 0.5000)
  recall      : 0.5000  (expected 0.5000)
  f1          : 0.5000  (expected 0.5000)
All metrics match hand-computed reference values ✓


In [5]:
# Larger synthetic dataset for sklearn comparison
torch.manual_seed(7)
N = 400
y_true_full = torch.randint(0, 2, (N,))
scores_full = torch.randn(N)  # raw scores / logits
# Bias the classifier: positive class gets higher mean score
scores_full = scores_full + 0.8 * (2 * y_true_full.float() - 1)
y_pred_full = (scores_full > 0.0).long()

acc_s  = accuracy(y_true_full, y_pred_full)
prec_s = precision_score_scratch(y_true_full, y_pred_full)
rec_s  = recall_score_scratch(y_true_full, y_pred_full)
f1_s   = f1_score_scratch(y_true_full, y_pred_full)

print(f"Scratch  | acc={acc_s:.4f}  prec={prec_s:.4f}  rec={rec_s:.4f}  f1={f1_s:.4f}")

if SKLEARN_AVAILABLE:
    yt = y_true_full.numpy()
    yp = y_pred_full.numpy()
    acc_sk  = accuracy_score(yt, yp)
    prec_sk = precision_score(yt, yp)
    rec_sk  = recall_score(yt, yp)
    f1_sk   = f1_score(yt, yp)
    print(f"sklearn  | acc={acc_sk:.4f}  prec={prec_sk:.4f}  rec={rec_sk:.4f}  f1={f1_sk:.4f}")
    assert abs(acc_s  - acc_sk)  < 1e-6, f"accuracy mismatch: {acc_s} vs {acc_sk}"
    assert abs(prec_s - prec_sk) < 1e-6, f"precision mismatch"
    assert abs(rec_s  - rec_sk)  < 1e-6, f"recall mismatch"
    assert abs(f1_s   - f1_sk)   < 1e-6, f"f1 mismatch"
    print("All metrics match sklearn ✓")
else:
    # sklearn not available: validate consistency of scratch metrics
    tp, fp, fn, tn = confusion_matrix_binary(y_true_full, y_pred_full)
    expected_prec = tp / (tp + fp)
    expected_rec  = tp / (tp + fn)
    expected_f1   = 2 * expected_prec * expected_rec / (expected_prec + expected_rec)
    assert abs(prec_s - expected_prec) < 1e-6
    assert abs(rec_s  - expected_rec)  < 1e-6
    assert abs(f1_s   - expected_f1)   < 1e-6
    print("Metrics internally consistent (sklearn unavailable) ✓")


Scratch  | acc=0.7750  prec=0.7864  rec=0.7788  f1=0.7826
Metrics internally consistent (sklearn unavailable) ✓


## Confusion matrix

A heatmap of TP, FP, FN, TN provides an at-a-glance summary of error patterns. Note that accuracy hides the distinction between FP and FN, which often have very different costs in production (e.g. a false negative in cancer screening vs a false positive).

In [6]:
def plot_confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, ax=None):
    """Plot a 2x2 confusion matrix heatmap."""
    tp, fp, fn, tn = confusion_matrix_binary(y_true, y_pred)
    cm = [[tn, fp], [fn, tp]]
    labels = [["TN", "FP"], ["FN", "TP"]]
    if ax is None:
        _, ax = plt.subplots()
    im = ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{labels[i][j]}\n{cm[i][j]}", ha="center", va="center", fontsize=13)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"])
    ax.set_yticklabels(["True 0", "True 1"])
    ax.set_title("Confusion Matrix")
    return ax


fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion_matrix(y_true_full, y_pred_full, ax=ax)
plt.tight_layout()
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_17378/2167129624.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Precision / Recall tradeoff

Threshold-dependent metrics (accuracy, precision, recall, F1) depend on the decision threshold applied to classifier scores. Sweeping the threshold traces the precision-recall curve without retraining.

- A **low threshold** classifies more samples as positive: recall goes up, precision goes down.
- A **high threshold** classifies fewer samples as positive: precision goes up, recall goes down.

The threshold that maximizes F1 is often a good default operating point.

In [7]:
thresholds = torch.linspace(-3, 3, 200)

prec_curve: list[float] = []
rec_curve:  list[float] = []
f1_curve:   list[float] = []

for thr in thresholds:
    y_pred_t = (scores_full > thr).long()
    prec_curve.append(precision_score_scratch(y_true_full, y_pred_t))
    rec_curve.append(recall_score_scratch(y_true_full, y_pred_t))
    f1_curve.append(f1_score_scratch(y_true_full, y_pred_t))

best_f1_idx = int(torch.tensor(f1_curve).argmax().item())
best_thr = thresholds[best_f1_idx].item()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# PR curve
axes[0].plot(rec_curve, prec_curve, lw=2)
axes[0].scatter(
    [rec_curve[best_f1_idx]], [prec_curve[best_f1_idx]],
    color="red", zorder=5, label=f"best F1 thr={best_thr:.2f}"
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall curve")
axes[0].legend()

# Metrics vs threshold
thr_np = thresholds.numpy()
axes[1].plot(thr_np, prec_curve, label="Precision")
axes[1].plot(thr_np, rec_curve, label="Recall")
axes[1].plot(thr_np, f1_curve, label="F1")
axes[1].axvline(x=best_thr, color="red", linestyle="--", label=f"best F1 @ {best_thr:.2f}")
axes[1].set_xlabel("threshold")
axes[1].set_title("Metrics vs threshold")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Best F1: {f1_curve[best_f1_idx]:.4f} at threshold {best_thr:.3f}")


Best F1: 0.7837 at threshold 0.196


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_17378/1123105922.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ROC curve and AUC

The Receiver Operating Characteristic (ROC) curve plots the true positive rate (recall) against the false positive rate (1 - specificity) across all thresholds. It is threshold-free and measures whether the classifier ranks positives above negatives.

$$\text{TPR} = \frac{TP}{TP + FN}, \quad \text{FPR} = \frac{FP}{FP + TN}$$

**AUC** (area under the ROC curve) has a probabilistic interpretation: it equals the probability that a randomly chosen positive sample receives a higher score than a randomly chosen negative sample. A random classifier has AUC = 0.5; a perfect classifier has AUC = 1.0.

In [8]:
def roc_curve_scratch(
    y_true: torch.Tensor, scores: torch.Tensor
) -> tuple[list[float], list[float], list[float]]:
    """Compute ROC curve by sweeping all unique score values as thresholds.

    Returns (fpr_list, tpr_list, thresholds).
    """
    # unique() only accepts sorted=True/False; flip to get descending order
    thrs = scores.unique(sorted=True).flip(0)
    fpr_list: list[float] = [0.0]
    tpr_list: list[float] = [0.0]
    thr_list: list[float] = [thrs[0].item() + 1]

    total_pos = int((y_true == 1).sum().item())
    total_neg = int((y_true == 0).sum().item())

    for thr in thrs:
        y_pred_t = (scores >= thr).long()
        tp = int(((y_pred_t == 1) & (y_true == 1)).sum().item())
        fp = int(((y_pred_t == 1) & (y_true == 0)).sum().item())
        fpr_list.append(fp / total_neg if total_neg > 0 else 0.0)
        tpr_list.append(tp / total_pos if total_pos > 0 else 0.0)
        thr_list.append(thr.item())

    fpr_list.append(1.0)
    tpr_list.append(1.0)
    return fpr_list, tpr_list, thr_list


def auc_trapezoidal(fpr: list[float], tpr: list[float]) -> float:
    """Area under ROC via trapezoidal rule."""
    total = 0.0
    for i in range(1, len(fpr)):
        dx = fpr[i] - fpr[i - 1]
        total += dx * (tpr[i] + tpr[i - 1]) / 2
    return abs(total)  # abs handles any order-flipping


fpr, tpr, thr_list = roc_curve_scratch(y_true_full, scores_full)
auc_scratch = auc_trapezoidal(fpr, tpr)
print(f"AUC (scratch trapezoidal): {auc_scratch:.4f}")

if SKLEARN_AVAILABLE:
    auc_sk = roc_auc_score(y_true_full.numpy(), scores_full.numpy())
    print(f"AUC (sklearn):             {auc_sk:.4f}")
    assert abs(auc_scratch - auc_sk) < 1e-3, f"AUC mismatch: {auc_scratch} vs {auc_sk}"
    print("AUC matches sklearn ✓")
else:
    # Sanity: AUC of a classifier that separates classes should be > 0.5
    assert auc_scratch > 0.5, f"Expected AUC > 0.5 for non-trivial classifier, got {auc_scratch}"
    print(f"AUC > 0.5 for non-trivial classifier (sklearn unavailable) ✓")

AUC (scratch trapezoidal): 0.8628
AUC > 0.5 for non-trivial classifier (sklearn unavailable) ✓


In [9]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f"ROC curve (AUC = {auc_scratch:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="random classifier")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_17378/3307460700.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## When accuracy is misleading

With 95% negatives and 5% positives, a classifier that predicts 'negative' for everything achieves 95% accuracy but catches zero positives. Precision, recall, and AUC expose this failure.

In [10]:
torch.manual_seed(99)
N_imb = 1000
y_imb = torch.cat([torch.ones(50, dtype=torch.long), torch.zeros(950, dtype=torch.long)])
y_pred_all_neg = torch.zeros(N_imb, dtype=torch.long)  # trivial all-negative classifier

acc_imb  = accuracy(y_imb, y_pred_all_neg)
prec_imb = precision_score_scratch(y_imb, y_pred_all_neg)
rec_imb  = recall_score_scratch(y_imb, y_pred_all_neg)
f1_imb   = f1_score_scratch(y_imb, y_pred_all_neg)

print(f"All-negative classifier on 5% positive data:")
print(f"  Accuracy:  {acc_imb:.3f}  ← looks great!")
print(f"  Precision: {prec_imb:.3f}")
print(f"  Recall:    {rec_imb:.3f}  ← catches zero positives")
print(f"  F1:        {f1_imb:.3f}")

assert acc_imb > 0.9, "Expected high accuracy for all-negative on imbalanced data"
assert rec_imb == 0.0, "All-negative classifier should have zero recall"
assert f1_imb == 0.0, "All-negative classifier should have zero F1"
print("Accuracy misleads under class imbalance; recall and F1 expose the failure ✓")


All-negative classifier on 5% positive data:
  Accuracy:  0.950  ← looks great!
  Precision: 0.000
  Recall:    0.000  ← catches zero positives
  F1:        0.000
Accuracy misleads under class imbalance; recall and F1 expose the failure ✓


## Takeaways

- **Accuracy** answers 'how often are we right?' but is misleading when classes are imbalanced.
- **Precision** = TP / (TP + FP): when we raise an alarm, how often is it real? Use when false positives are costly.
- **Recall** (TPR, sensitivity) = TP / (TP + FN): how many real cases did we catch? Use when false negatives are costly.
- **F1** is the harmonic mean of precision and recall. It punishes systems that achieve one at the expense of the other.
- **ROC-AUC** is threshold-free: it measures whether the classifier ranks positives above negatives. AUC = 0.5 is random; AUC = 1.0 is perfect. It is less informative when the positive class is rare — in that case, PR-AUC focuses on positive predictions.
- Threshold sweeping shows the precision-recall tradeoff without retraining. The optimal threshold depends on the relative cost of false positives vs false negatives.
- Metrics are not losses: they measure task success on held-out data. The training loss (e.g. cross-entropy) is a differentiable proxy for the metric you actually care about.